# ETL Process Prototype

In this notebook, we are going to build a functional prototype of the Extract, Transform, and Load (ETL) pipeline. This logic will subsequently be modularized into the `src/` Python scripts in a professional manner. 

We will follow the Star Schema designed in `diagrams/README.md` and the initial transformation roadmap outlined in `notebooks/eda.ipynb`.
For the purpose of fast prototyping and testing, the Load step will target a local `sqlite3` database instead of PostgreSQL. 


## 1. Extract
Read the raw `candidates.csv` file using the appropriate non-standard semicolon separator.


In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Extract
file_path = '../data/raw/candidates.csv'
df_raw = pd.read_csv(file_path, sep=';')

print(f"Extracted {len(df_raw)} records.")
display(df_raw.head())


Extracted 50000 records.


,First Name,Last Name,Email,Application Date,Country,YOE,Seniority,Technology,Code Challenge Score,Technical Interview Score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7


## 2. Transform: Cleansing & Core Logic
We need to:
1. Drop exact duplicates.
2. Standardize column names (snake_case is preferable for databases).
3. Enforce data types.
4. Inject the `is_hired` business rule (`Code Challenge Score >= 7` AND `Technical Interview Score >= 7`).


In [2]:
df = df_raw.copy()

# Drop duplicates
df = df.drop_duplicates()

# Standardize column names
df.columns = [
    'first_name', 'last_name', 'email', 'application_date', 'country', 
    'yoe', 'seniority', 'technology', 'score_code_challenge', 
    'score_technical_interview'
]

# Enforce Types
df['application_date'] = pd.to_datetime(df['application_date'])

# Inject Business logic
df['is_hired'] = np.where(
    (df['score_code_challenge'] >= 7) & (df['score_technical_interview'] >= 7),
    1, 0
)

print(f"Data shape after initial cleansing: {df.shape}")
display(df.head())


Data shape after initial cleansing: (50000, 11)


,first_name,last_name,email,application_date,country,yoe,seniority,technology,score_code_challenge,score_technical_interview,is_hired
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3,0
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10,0
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9,1
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1,0
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7,1


## 3. Transform: Dimension Generation
We must separate the cleansed data into our dimension tables, extract unique values, and generate Surrogate Keys (integer IDs).

**Dimensions according to our design:**
- `dim_candidates` (first_name, last_name, email)
- `dim_jobs` (technology, seniority)
- `dim_locations` (country)
- `dim_times` (derived from application_date)


In [3]:
# --- 3.1 dim_candidates ---
dim_candidates = df[['first_name', 'last_name', 'email']].drop_duplicates().reset_index(drop=True)
dim_candidates.insert(0, 'candidate_id', dim_candidates.index + 1)
print(f"dim_candidates shape: {dim_candidates.shape}")

# --- 3.2 dim_jobs ---
dim_jobs = df[['technology', 'seniority']].drop_duplicates().reset_index(drop=True)
dim_jobs.insert(0, 'job_id', dim_jobs.index + 1)
print(f"dim_jobs shape: {dim_jobs.shape}")

# --- 3.3 dim_locations ---
dim_locations = df[['country']].drop_duplicates().reset_index(drop=True)
dim_locations.insert(0, 'location_id', dim_locations.index + 1)
print(f"dim_locations shape: {dim_locations.shape}")

# --- 3.4 dim_times ---
dim_times = df[['application_date']].drop_duplicates().reset_index(drop=True)
dim_times['full_date'] = dim_times['application_date'].dt.strftime('%Y-%m-%d')
dim_times['year'] = dim_times['application_date'].dt.year
dim_times['month'] = dim_times['application_date'].dt.month
dim_times['day'] = dim_times['application_date'].dt.day
dim_times['quarter'] = dim_times['application_date'].dt.quarter
dim_times = dim_times[['full_date', 'year', 'month', 'day', 'quarter']]
dim_times.insert(0, 'date_id', dim_times.index + 1)
print(f"dim_times shape: {dim_times.shape}")


dim_candidates shape: (50000, 4)
dim_jobs shape: (168, 3)
dim_locations shape: (244, 2)
dim_times shape: (1646, 6)


## 4. Transform: Fact Table Generation
Finally, we build `fact_applications`. We use Pandas `merge()` to map the source string/date values back to the generated Surrogate Keys of our dimension tables. We will also include our measures (`score_code_challenge`, `score_technical_interview`, `is_hired`).


In [4]:
# Create a temporary column to match dates as strings so merging works seamlessly
df['full_date'] = df['application_date'].dt.strftime('%Y-%m-%d')

fact_applications = df.copy()

# Merge Candidates to get candidate_id
fact_applications = fact_applications.merge(dim_candidates, on=['first_name', 'last_name', 'email'], how='left')

# Merge Jobs to get job_id
fact_applications = fact_applications.merge(dim_jobs, on=['technology', 'seniority'], how='left')

# Merge Locations to get location_id
fact_applications = fact_applications.merge(dim_locations, on=['country'], how='left')

# Merge Times to get date_id
fact_applications = fact_applications.merge(dim_times[['date_id', 'full_date']], on=['full_date'], how='left')

# Keep only the Surrogate Keys and Measures
fact_columns = [
    'candidate_id', 'date_id', 'job_id', 'location_id', 
    'score_code_challenge', 'score_technical_interview', 'is_hired'
]
fact_applications = fact_applications[fact_columns]

# Add the final Application ID (PK for the fact table, if desired, though often identity in the DB)
fact_applications.insert(0, 'application_id', fact_applications.index + 1)

print(f"fact_applications shape: {fact_applications.shape}")
display(fact_applications.head())


fact_applications shape: (50000, 8)


,application_id,candidate_id,date_id,job_id,location_id,score_code_challenge,score_technical_interview,is_hired
0,1,1,1,1,1,3,3,0
1,2,2,2,1,2,2,10,0
2,3,3,3,2,3,10,9,1
3,4,4,4,3,4,7,1,0
4,5,5,5,4,5,9,7,1


## 5. Load (Prototyping with SQLite)
For the prototype phase, we will load these DataFrames into a local SQLite database file `../data/processed/warehouse.db`. In the production code (`src/load.py`), we will adapt this logic using `SQLAlchemy` and `psycopg2` to write to PostgreSQL.


In [5]:
# Create the directory if it doesn't exist
import os
os.makedirs('../data/processed', exist_ok=True)

db_path = '../data/processed/warehouse.db'

# Open a connection to SQLite
conn = sqlite3.connect(db_path)

try:
    # Write dimensions (using if_exists='replace' simplifies re-running the notebook)
    dim_candidates.to_sql('dim_candidates', conn, if_exists='replace', index=False)
    dim_jobs.to_sql('dim_jobs', conn, if_exists='replace', index=False)
    dim_locations.to_sql('dim_locations', conn, if_exists='replace', index=False)
    dim_times.to_sql('dim_times', conn, if_exists='replace', index=False)
    
    # Write fact
    fact_applications.to_sql('fact_applications', conn, if_exists='replace', index=False)
    print("Data successfully loaded into SQLite!")
    
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    conn.close()


Data successfully loaded into SQLite!


## 6. Verification
Run a quick test query against our newly created SQLite Star Schema to verify relationships are working.


In [6]:
# Test Query: Hires By Technology
query = """
SELECT
    j.technology,
    SUM(f.is_hired) as total_hires
FROM fact_applications f
JOIN dim_jobs j ON f.job_id = j.job_id
GROUP BY j.technology
ORDER BY total_hires DESC
LIMIT 5;
"""

conn = sqlite3.connect(db_path)
test_results = pd.read_sql(query, conn)
conn.close()

print("Top 5 Technologies by Hires:")
display(test_results)


Top 5 Technologies by Hires:


,technology,total_hires
0,Game Development,519
1,DevOps,495
2,System Administration,293
3,Development - CMS Backend,284
4,Database Administration,282
